In [19]:
## Mirrors the cellPLM- and scGPT-embedding-wrapper notebooks: load a dataset,
## tokenize it for Geneformer, run zero-shot inference with the V2-104M
## checkpoint, and write the per-cell <cls> embedding to a small h5ad keyed by
## obs_names so benchmark.ipynb can align it positionally-independently.

In [20]:
import sys, os
sys.path.insert(0, os.path.abspath('../../models/Geneformer'))

import warnings
warnings.filterwarnings('ignore')

import pickle
import random
import shutil
from pathlib import Path

import numpy as np
import torch
import anndata as ad
import geneformer
from geneformer import TranscriptomeTokenizer, EmbExtractor

In [ ]:
DATA_PATH    = Path('../../data/cellPLM/data/gse155468.h5ad')
GF_REPO_DIR  = Path('../../models/Geneformer')
MODEL_DIR    = GF_REPO_DIR / 'Geneformer-V2-104M'    # V2 architecture, 104M params, d=512
MODEL_VERSION = 'V2'
DEVICE       = 'cuda:0'
SEED         = 42

# Working dir for tokenizer intermediates (a temp h5ad + a .dataset). Kept
# next to this notebook so reruns are easy to inspect; cleaned at the end.
WORK_DIR  = Path('./_geneformer_workdir').resolve()
WORK_DIR.mkdir(exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ.setdefault('CUDA_VISIBLE_DEVICES', DEVICE.split(':')[1] if ':' in DEVICE else '0')

'0'

In [22]:
# Load the source dataset and prepare the two metadata fields Geneformer needs:
#   var['ensembl_id'] — gene Ensembl IDs (used for tokenization vocab lookup)
#   obs['n_counts']   — total UMI counts per cell (used for normalization)
# We also stash an `original_idx` per cell so we can recover the source ordering
# after Geneformer's tokenizer/extractor reorder cells internally.
data = ad.read_h5ad(DATA_PATH)
data.obs_names_make_unique()

# gse155468.h5ad uses HGNC gene SYMBOLS in var_names with no var columns.
# Geneformer wants Ensembl IDs, so convert via Geneformer's bundled mapping.
gene_name_id_pkl = Path(geneformer.__file__).parent / 'gene_name_id_dict_gc104M.pkl'
with open(gene_name_id_pkl, 'rb') as f:
    gene_name_to_id = pickle.load(f)

ensembl_ids = [gene_name_to_id.get(g, '') for g in data.var_names]
data.var['ensembl_id'] = ensembl_ids
n_mapped = sum(bool(e) for e in ensembl_ids)
print(f'mapped {n_mapped}/{data.n_vars} gene symbols to Ensembl IDs')

# nCount_RNA already holds total UMIs per cell — Geneformer reads obs['n_counts'].
data.obs['n_counts']     = data.obs['nCount_RNA'].astype('int64')
data.obs['original_idx'] = np.arange(data.n_obs, dtype=np.int64)

tmp_h5ad = WORK_DIR / 'gse155468.h5ad'
data.write_h5ad(tmp_h5ad)
print(f'wrote {tmp_h5ad}, shape={data.shape}')

mapped 11136/12382 gene symbols to Ensembl IDs
wrote /data/benchmark/embd_clustering/Geneformer-embedding-wrapper/_geneformer_workdir/gse155468.h5ad, shape=(48082, 12382)


In [23]:
# Tokenize the dataset into Geneformer's rank-value-encoded format.
# Output is a HuggingFace `datasets.Dataset` saved to disk as `*.dataset`.
# custom_attr_name_dict propagates `original_idx` through tokenization so we
# can reorder embeddings back to data.obs_names after extraction.
tk = TranscriptomeTokenizer(
    custom_attr_name_dict={'original_idx': 'original_idx'},
    nproc=4,
    model_version=MODEL_VERSION,
)
tk.tokenize_data(
    data_directory=WORK_DIR,
    output_directory=WORK_DIR,
    output_prefix='gse155468',
    file_format='h5ad',
)
tokenized_path = WORK_DIR / 'gse155468.dataset'
print(f'tokenized -> {tokenized_path}')

Tokenizing /data/benchmark/embd_clustering/Geneformer-embedding-wrapper/_geneformer_workdir/gse155468.h5ad
/data/benchmark/embd_clustering/Geneformer-embedding-wrapper/_geneformer_workdir/gse155468.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.
tokenized -> /data/benchmark/embd_clustering/Geneformer-embedding-wrapper/_geneformer_workdir/gse155468.dataset


In [24]:
# Run inference and pull <cls>-token cell embeddings for every cell.
# Notes:
#   max_ncells=None       — emit embeddings for ALL cells (default is 1000!).
#   emb_mode='cls'        — V2 default; uses the dedicated <cls> token vector.
#   emb_label=['original_idx'] — carry the source-order index into the output
#                          dataframe so we can re-sort to data.obs_names.
embex = EmbExtractor(
    model_type='Pretrained',
    emb_mode='cls',
    emb_layer = 0, 
    emb_label=['original_idx'],
    max_ncells=None,
    forward_batch_size=64,
    nproc=4,
    model_version=MODEL_VERSION,
)
embs_df = embex.extract_embs(
    model_directory=str(MODEL_DIR),
    input_data_file=str(tokenized_path),
    output_directory=str(WORK_DIR),
    output_prefix='gse155468_embs',
)
print(f'extracted embeddings: shape={embs_df.shape}, columns[:5]={list(embs_df.columns[:5])}')

model_version selected as V1 so changing emb_mode from 'cls' to 'cell' as V1 models do not have a <cls> token.
100%|██████████| 752/752 [02:07<00:00,  5.89it/s]


extracted embeddings: shape=(48082, 257), columns[:5]=[0, 1, 2, 3, 4]


In [25]:
# Re-sort by original_idx so row i corresponds to data.obs_names[i].
# Drop label columns; what remains are the embedding dimensions.
embs_sorted = embs_df.sort_values('original_idx').reset_index(drop=True)
label_cols  = [c for c in embs_sorted.columns if not isinstance(c, (int, np.integer))]
embedding   = embs_sorted.drop(columns=label_cols).to_numpy().astype(np.float32)
print('embedding shape:', embedding.shape, 'dtype:', embedding.dtype)
assert embedding.shape[0] == data.n_obs, (embedding.shape, data.n_obs)

embedding shape: (48082, 256) dtype: float32


In [26]:
# Match cellPLM-/scGPT-embedding-wrapper output layout so benchmark.ipynb can
# align by obs_names: X = (n_cells, d) float32, obs_names = original cell IDs,
# var_names = ['dim_0', 'dim_1', ...].
embedding_adata = ad.AnnData(X=embedding)
embedding_adata.obs_names = data.obs_names
embedding_adata.var_names = [f'dim_{i}' for i in range(embedding.shape[1])]
embedding_adata.write_h5ad('gse155468_embedding.h5ad')
print('wrote gse155468_embedding.h5ad', embedding_adata.shape)

# Tokenizer/extractor scratch is a few hundred MB; remove unless you want to
# inspect intermediates. Comment this out to keep WORK_DIR for debugging.
shutil.rmtree(WORK_DIR, ignore_errors=True)

wrote gse155468_embedding.h5ad (48082, 256)
